In [18]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import random
from sklearn.metrics import classification_report

In [19]:
iris = load_iris()

In [20]:
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.DataFrame(iris.target, columns=['y'])

In [21]:
df = pd.DataFrame(X)
df['y'] = y

In [22]:
df.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),y
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


In [23]:
class Perceptron:
    def __init__(self, df):
        x, y = df.loc[:, df.columns!='y'], df['y']
        x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.33)
        self.x_train = self.normalize(x_train)
        self.x_train.insert(0,'x0',1)
        self.x_test = x_test
        self.x_test.insert(0,'x0',1)
        self.y_train = y_train
        self.y_test = y_test
        self.d_features = len(x.columns)
        self.n_datapoints = self.x_train.shape[0]
        self.l_classes = len(set(y))
        self.Ws = [np.array([0 for _ in range(self.d_features+1)]) for _ in range(self.l_classes)]

    def normalize(self, df):
        df_z_scaled = df.copy() 
  
        for column in df_z_scaled.columns: 
            df_z_scaled[column] = (df_z_scaled[column] - df_z_scaled[column].mean()) / df_z_scaled[column].std()

        return df_z_scaled
    
    def display_attr(self):
        # print(self.Ws)
        for i, elt in enumerate(self.Ws):
            print(i, elt)

    def softmax(self, index, output_neuron_values):
        numerator = np.exp(output_neuron_values[index])
        denominator = 0

        for val in output_neuron_values:
            denominator += np.exp(val)

        return numerator/denominator

    def argmax(self, output_neuron_values):
        maxidx = -1
        maxval = -1e12

        for i, elt in enumerate(output_neuron_values):
            if elt>maxval:
                maxval = elt
                maxidx = i

        return maxidx

    def build_model(self):
        n = self.n_datapoints
        d = self.d_features
        l = self.l_classes

        Wolds = self.Ws[:]
        Wnews = self.Ws[:]
        epochs = 0
        # eta = random.choice(np.linspace(0, 0.1, 1000))
        eta = 0.001
        lb = 0.1
        
        x = self.x_train
        y = self.y_train

        printer_steps = set(int(np.floor(i)) for i in np.linspace(0, n, 50))

        while True:
            if epochs!=0:
                for i, elt in enumerate(Wolds):
                    Wnews[i] = elt

            if epochs%1000==0:
                print(f"Running Epoch {epochs+1} ", end='')

            for i in range(n):
                if epochs%1000==0 and i in printer_steps:
                    print("==", end='')

                x_vec = x.iloc[i]
                y_class = y.iloc[i]
                y_true = 0

                output_neuron_values = []

                for j in range(l):
                    Wj = Wolds[j]
                    WjTx = Wj.T.dot(x_vec)
                    output_neuron_values.append(WjTx)

                for j in range(l):
                    y_true = 1 if j==y_class else 0

                    softmax_probability = self.softmax(j, output_neuron_values)
                    Wolds[j] = Wolds[j] - eta*((softmax_probability-y_true)*np.array(x_vec)+lb*Wolds[j])

            if epochs%1000==0:
                print(">")
                for i, elt in enumerate(Wolds):
                    print(f"Old vs New Weight of Hyperplane{i+1}")
                    print(f"Wold {i} {Wolds[i]}, Wnew {i} {Wnews[i]}")

            brkflg = True
            epochs += 1

            # print(Wolds, Wnews)
            for i, elt in enumerate(Wolds):
                if not np.allclose(Wolds[i],Wnews[i]):
                    brkflg = False

            if brkflg:
                break

        self.Ws = Wnews

        print(f"Model training Successful at epochs {epochs}")
        print(f"Final Weights: ")
        for i, wi in enumerate(self.Ws):
            print(i, wi)

    def test_model(self):
        x = self.x_test
        n = x.shape[0]
        Ws = self.Ws
        l = self.l_classes
        y_trues = np.array(self.y_test)
        y_preds = []

        for i in range(n):
            x_vec = x.iloc[i]
            y_true = 0

            output_neuron_values = []

            for j in range(l):
                Wj = Ws[j]
                WjTx = Wj.T.dot(x_vec)
                output_neuron_values.append(WjTx)

            y_preds.append(self.argmax(output_neuron_values))

        y_preds = np.array(y_preds)

        print(classification_report(y_trues, y_preds))

In [24]:
x, y = df.loc[:, df.columns!='y'], df['y']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.33)

In [25]:
x_train

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
140,6.7,3.1,5.6,2.4
145,6.7,3.0,5.2,2.3
63,6.1,2.9,4.7,1.4
40,5.0,3.5,1.3,0.3
38,4.4,3.0,1.3,0.2
...,...,...,...,...
73,6.1,2.8,4.7,1.2
14,5.8,4.0,1.2,0.2
130,7.4,2.8,6.1,1.9
26,5.0,3.4,1.6,0.4


In [26]:
p1 = Perceptron(df)

In [27]:
p1.display_attr()

0 [0 0 0 0 0]
1 [0 0 0 0 0]
2 [0 0 0 0 0]


In [28]:
np.exp(1)

2.718281828459045

In [29]:
p1.build_model()

Running Epoch 1 ==================================================================================================>
Old vs New Weight of Hyperplane1
Wold 0 [-0.00326785 -0.03145504  0.02378336 -0.03961904 -0.03819376], Wnew 0 [0 0 0 0 0]
Old vs New Weight of Hyperplane2
Wold 1 [ 0.00061789  0.00063778 -0.01925712  0.00653562  0.00380063], Wnew 1 [0 0 0 0 0]
Old vs New Weight of Hyperplane3
Wold 2 [ 0.00264996  0.03081726 -0.00452625  0.03308343  0.03439314], Wnew 2 [0 0 0 0 0]
Model training Successful at epochs 671
Final Weights: 
0 [-0.24504229 -0.41963093  0.44845075 -0.6451891  -0.60323826]
1 [ 0.37021908  0.01149487 -0.3872926   0.06233661 -0.08299804]
2 [-0.12517679  0.40813606 -0.06115815  0.58285249  0.6862363 ]


In [31]:
p1.test_model()

              precision    recall  f1-score   support

           0       0.00      0.00      0.00        20
           1       0.00      0.00      0.00        16
           2       0.28      1.00      0.44        14

    accuracy                           0.28        50
   macro avg       0.09      0.33      0.15        50
weighted avg       0.08      0.28      0.12        50



/home/karthikmsd/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/karthikmsd/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/karthikmsd/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [32]:
p1.Ws

[array([-0.24504229, -0.41963093,  0.44845075, -0.6451891 , -0.60323826]),
 array([ 0.37021908,  0.01149487, -0.3872926 ,  0.06233661, -0.08299804]),
 array([-0.12517679,  0.40813606, -0.06115815,  0.58285249,  0.6862363 ])]

In [33]:
x = p1.x_test
y = p1.y_test
n = x.shape[0]
Ws = p1.Ws
l = p1.l_classes
y_trues = np.array(p1.y_test)
y_preds = []
y_pred_probs = []

for i in range(n):
    x_vec = x.iloc[i]
    y_true = 0
    y_class = y.iloc[i]

    output_neuron_values = []

    for j in range(l):
        Wj = Ws[j]
        WjTx = Wj.T.dot(x_vec)
        output_neuron_values.append(WjTx)

    y_preds.append(p1.argmax(output_neuron_values))

    softmax_probabilities = []

    for j in range(l):
        y_true = 1 if j==y_class else 0
        softmax_probability = p1.softmax(j, output_neuron_values)
        softmax_probabilities.append(softmax_probability)

    y_pred_probs.append(softmax_probabilities)

y_preds = np.array(y_preds)
y_pred_probs = np.array(y_pred_probs)

print(classification_report(y_trues, y_preds))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00        20
           1       0.00      0.00      0.00        16
           2       0.28      1.00      0.44        14

    accuracy                           0.28        50
   macro avg       0.09      0.33      0.15        50
weighted avg       0.08      0.28      0.12        50



/home/karthikmsd/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/karthikmsd/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/karthikmsd/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [34]:
from sklearn.metrics import roc_auc_score
# 1v1
roc_auc_score(y_trues, y_pred_probs, multi_class='ovo')

0.8199404761904763

In [35]:
roc_auc_score(y_trues, y_pred_probs, multi_class='ovr')

0.792454481792717